In [2]:
# ============================================================
# Fold-by-fold comparison:
# CNN-AE vs. classical baselines
# ============================================================

from pathlib import Path
import pandas as pd

REPO_ROOT = Path("..").resolve()

TABLE_DIR = REPO_ROOT / "reports" / "tables"

AE_METRICS_PATH = (
    TABLE_DIR
    / "metrics_turning_ae_grouped_cv.csv"
)

BASELINE_METRICS_PATH = (
    TABLE_DIR
    / "metrics_turning_grouped_cv_rgb.csv"
)


# ------------------------------------------------------------
# Load fold-level results
# ------------------------------------------------------------

ae_metrics = pd.read_csv(
    AE_METRICS_PATH
)

baseline_metrics = pd.read_csv(
    BASELINE_METRICS_PATH
)


# ------------------------------------------------------------
# Select AE score used for the comparison
# ------------------------------------------------------------

ae_fold = (
    ae_metrics[
        ae_metrics["score"] == "global_mae"
    ][
        [
            "outer_fold",
            "pr_auc",
            "f1",
            "precision",
            "recall",
        ]
    ]
    .copy()
)

ae_fold["method"] = "CNN-AE (global MAE)"


# ------------------------------------------------------------
# Select baseline methods
# ------------------------------------------------------------

baseline_fold = (
    baseline_metrics[
        baseline_metrics["method"].isin(
            [
                "isolation_forest_rgb_features",
                "one_class_svm_rgb_features",
                "pca_rgb_reconstruction",
            ]
        )
    ][
        [
            "outer_fold",
            "method",
            "pr_auc",
            "f1",
            "precision",
            "recall",
        ]
    ]
    .copy()
)


method_names = {
    "isolation_forest_rgb_features":
        "Isolation Forest",
    "one_class_svm_rgb_features":
        "One-Class SVM",
    "pca_rgb_reconstruction":
        "PCA Reconstruction",
}

baseline_fold["method"] = (
    baseline_fold["method"]
    .replace(method_names)
)


# ------------------------------------------------------------
# Combine results
# ------------------------------------------------------------

fold_comparison_long = pd.concat(
    [
        ae_fold,
        baseline_fold,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# PR-AUC fold-by-fold table
# ------------------------------------------------------------

fold_comparison_pr_auc = (
    fold_comparison_long
    .pivot(
        index="outer_fold",
        columns="method",
        values="pr_auc",
    )
    .reset_index()
)


# Preferred column order
column_order = [
    "outer_fold",
    "CNN-AE (global MAE)",
    "Isolation Forest",
    "One-Class SVM",
    "PCA Reconstruction",
]

fold_comparison_pr_auc = (
    fold_comparison_pr_auc[
        [
            col
            for col in column_order
            if col in fold_comparison_pr_auc.columns
        ]
    ]
)


# ------------------------------------------------------------
# Difference: Isolation Forest minus CNN-AE
# ------------------------------------------------------------

if (
    "Isolation Forest" in fold_comparison_pr_auc.columns
    and
    "CNN-AE (global MAE)" in fold_comparison_pr_auc.columns
):

    fold_comparison_pr_auc[
        "IF_minus_AE"
    ] = (
        fold_comparison_pr_auc[
            "Isolation Forest"
        ]
        -
        fold_comparison_pr_auc[
            "CNN-AE (global MAE)"
        ]
    )


# ------------------------------------------------------------
# Prepare publication table
# ------------------------------------------------------------

fold_comparison_publication = (
fold_comparison_pr_auc
.rename(
columns={
"outer_fold": "Outer Fold",
}
)
.copy()
)

publication_columns = [
"Outer Fold",
"CNN-AE (global MAE)",
"Isolation Forest",
"One-Class SVM",
"PCA Reconstruction",
]

fold_comparison_publication = (
fold_comparison_publication[
publication_columns
]
)

# Ensure integer fold numbers
fold_comparison_publication["Outer Fold"] = (
fold_comparison_publication["Outer Fold"]
.astype(int)
)

# ------------------------------------------------------------
# Export LaTeX table
# ------------------------------------------------------------


FOLD_COMPARISON_TEX_PATH = (
TABLE_DIR
/ "turning_grouped_cv_fold_comparison.tex"
)

fold_comparison_publication.to_latex(
FOLD_COMPARISON_TEX_PATH,
index=False, # removes the first/index column
float_format="%.4f",
escape=True,
column_format="lcccc",
)

print(
f"Wrote {FOLD_COMPARISON_TEX_PATH}"
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(
    fold_comparison_pr_auc.round(4)
)


# ------------------------------------------------------------
# Mean across folds as sanity check
# ------------------------------------------------------------

print("\nMean PR-AUC across folds:")

display(
    fold_comparison_long
    .groupby("method")["pr_auc"]
    .agg(["mean", "std"])
    .sort_values(
        "mean",
        ascending=False,
    )
    .round(4)
)

Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\turning_grouped_cv_fold_comparison.tex


method,outer_fold,CNN-AE (global MAE),Isolation Forest,One-Class SVM,PCA Reconstruction,IF_minus_AE
0,1,1.0000,0.9914,0.9838,0.9098,-0.0086
1,2,0.5170,0.6894,0.6962,0.6187,0.1723
2,3,0.8672,0.9421,0.6406,0.5198,0.0749
3,4,0.9758,1.0000,0.8788,0.9473,0.0242
4,5,0.7053,0.7870,0.5691,0.6223,0.0817



Mean PR-AUC across folds:


,mean,std
method,,
Isolation Forest,0.8820,0.1375
CNN-AE (global MAE),0.8131,0.2022
One-Class SVM,0.7537,0.1723
PCA Reconstruction,0.7236,0.1921
